# Step 3: Causal, Leakage-Safe Feature Engineering

## 1. Objective

Build a feature set for fraud detection that could, in principle, be computed **before** a transaction is approved -- not one that only looks good because it has access to information that would not exist yet in a real system. All the actual feature-construction logic lives in `src/features.py` and `src/feature_definitions.py`; this notebook demonstrates and validates those reusable functions rather than reimplementing the pipeline inline.

## 2. Prediction-point definition

We assume the model scores a transaction the instant it is **requested**, before it is processed.

**Available at decision time:** `step`, `type`, `amount`, `nameOrig`, `nameDest`, `oldbalanceOrg`, `oldbalanceDest`, and any aggregate of a customer's **strictly earlier** transactions.

**Never available for the current transaction:** `newbalanceOrig`, `newbalanceDest` (post-transaction state), `isFlaggedFraud` (a rule output, not a transaction attribute), `isFraud` (the target).

This is enforced in code, not just by convention -- `newbalanceOrig`, `newbalanceDest`, and `isFlaggedFraud` are not used as inputs to any feature calculation in `src/features.py` (they appear only in explanatory docstrings warning against their use), and `isFraud` is only ever passed through as the preserved target column.

In [ ]:
import sys
sys.path.append("..")

import inspect
import time
import numpy as np
import pandas as pd

from src.data_utils import load_raw_data
from src import features as feat_module
from src.features import (
    build_feature_dataset,
    get_model_feature_columns,
    create_transaction_features,
    create_time_features,
    create_balance_features,
    create_sender_history_features,
    create_receiver_history_features,
    create_velocity_features,
)
from src.feature_definitions import FEATURE_DEFINITIONS

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

df = load_raw_data()
df.shape

## 3. Dataset ordering investigation

Before trusting any "previous row" logic, we need to know whether the file is sorted by time and whether rows sharing a `step` have any reliable sub-order.

In [ ]:
print("Sorted by step ascending:", df["step"].is_monotonic_increasing)

same_step_orig = df.groupby(["nameOrig", "step"]).size()
same_step_dest = df.groupby(["nameDest", "step"]).size()
print(f"(nameOrig, step) pairs with >1 transaction: {(same_step_orig > 1).sum()}")
print(f"(nameDest, step) pairs with >1 transaction: {(same_step_dest > 1).sum()}")
print("No sub-step timestamp/sequence column exists in the raw data.")

**Conclusion:** the file is globally sorted by `step`, but thousands of `(nameOrig, step)` and `(nameDest, step)` pairs tie on the exact same step with no way to know which happened first within that hour. **Design decision:** every historical/velocity feature in `src/features.py` uses only STRICTLY EARLIER steps (never a same-step neighboring row), via `pd.merge_asof(..., direction="backward")` on `step - offset`. This means two transactions sharing a step get identical history (based only on earlier steps) -- a conservative but defensible causal guarantee that doesn't depend on an unverifiable row-order assumption. This is verified with a controlled test in section 10b below.

## 4. Transaction features (Group A)

In [ ]:
txn_feats = create_transaction_features(df)
txn_feats.head()

## 5. Time features (Group B)

Derived purely from PaySim's **simulated** `step` -- not a real calendar timestamp.

In [ ]:
time_feats = create_time_features(df)
time_feats.head()

## 6. Pre-transaction balance features (Group C)

Uses only `oldbalanceOrg`/`oldbalanceDest`. The current row's `newbalanceOrig`/`newbalanceDest` are not used as inputs to this or any other feature calculation.

In [ ]:
balance_feats = create_balance_features(df)
balance_feats.head()

## 7. Historical sender features (Group D)

The most important group. Built via: (1) aggregate to one row per `(nameOrig, step)` using vectorized groupby aggregations, (2) take an inclusive cumulative sum/max/min per sender ordered by step, (3) `merge_asof` each transaction against that table at `step - 1` (strictly earlier). No Python row loops.

In [ ]:
t0 = time.time()
sender_table = feat_module._build_sender_step_table(df)
sender_feats = create_sender_history_features(df, sender_table)
print(f"Built in {time.time()-t0:.1f}s")
sender_feats.describe().T

## 8. Historical receiver features (Group E)

Same causal mechanism, keyed on `nameDest` instead of `nameOrig`.

In [ ]:
t0 = time.time()
receiver_table = feat_module._build_receiver_step_table(df)
receiver_feats = create_receiver_history_features(df, receiver_table)
print(f"Built in {time.time()-t0:.1f}s")
receiver_feats.describe().T

## 9. Velocity features (Group F)

`sender_transactions_previous_K_steps` = (sender's cumulative prior count as of `step-1`) minus (as of `step-1-K`), i.e. transactions strictly within the K steps immediately before the current one. K in {1, 6, 24}. Reuses the same per-step cumulative tables built above -- no recomputation.

In [ ]:
t0 = time.time()
velocity_feats = create_velocity_features(df, sender_table, receiver_table)
print(f"Built in {time.time()-t0:.1f}s")
velocity_feats.describe().T

## Building the full dataset

`build_feature_dataset()` orchestrates all six groups above, building the sender/receiver step tables once and reusing them for both the history and velocity groups (avoiding duplicate work).

In [ ]:
t0 = time.time()
feature_df = build_feature_dataset(df)
print(f"Full feature dataset built in {time.time()-t0:.1f}s -- shape {feature_df.shape}")

model_cols = get_model_feature_columns(feature_df)
print(f"\nModel-ready feature columns ({len(model_cols)}):")
for c in model_cols:
    print(" -", c)

## 10. Leakage validation

Explicit checks per the Step 3 requirements -- printed, not just claimed.

In [ ]:
# 1) First transaction per sender -> prior_sender_transaction_count == 0
first_txn_mask = ~df["nameOrig"].duplicated(keep="first")
check1 = (feature_df.loc[first_txn_mask, "prior_sender_transaction_count"] == 0).all()
print(f"[1] First-transaction senders have prior_sender_transaction_count == 0: {check1} (n={first_txn_mask.sum():,})")

# 2) First transaction per receiver -> prior_receiver_transaction_count == 0
first_dest_mask = ~df["nameDest"].duplicated(keep="first")
check2 = (feature_df.loc[first_dest_mask, "prior_receiver_transaction_count"] == 0).all()
print(f"[2] First-transaction receivers have prior_receiver_transaction_count == 0: {check2} (n={first_dest_mask.sum():,})")

# 3) Historical totals exclude the current transaction (spot check a real repeat sender)
repeat_senders = df["nameOrig"].value_counts()
sample_sender = repeat_senders[repeat_senders > 2].index[0]
sender_rows = df[df["nameOrig"] == sample_sender].sort_values("step")
sender_feat_rows = feature_df.loc[sender_rows.index].sort_values("step")
manual_prior_total = sender_rows["amount"].cumsum().shift(1).fillna(0)
check3 = np.allclose(sender_feat_rows["prior_sender_total_amount"].values, manual_prior_total.values)
print(f"[3] Historical totals exclude current transaction (spot check on sender {sample_sender}): {check3}")

# 4) A transaction at step X never uses a transaction from step > X
tsl = feature_df["prior_sender_time_since_last_transaction"]
check4 = ((tsl == -1) | (tsl >= 1)).all()
print(f"[4] prior_sender_time_since_last_transaction is always -1 (no history) or >=1 (strictly earlier step): {check4}")

# 5, 6 & 8) isFraud / isFlaggedFraud / newbalanceOrig / newbalanceDest are not used as inputs to any feature calculation
forbidden = ["isFraud", "isFlaggedFraud", "newbalanceOrig", "newbalanceDest"]
feature_functions = [
    feat_module.create_transaction_features, feat_module.create_time_features,
    feat_module.create_balance_features, feat_module.create_sender_history_features,
    feat_module.create_receiver_history_features, feat_module.create_velocity_features,
    feat_module._build_step_level_table, feat_module._first_occurrence_counts, feat_module._asof_lookup,
]
all_clean = {}
for fn in feature_functions:
    src = inspect.getsource(fn)
    all_clean[fn.__name__] = [c for c in forbidden if c in src]
print("[5/6/8] Forbidden-column mentions found by a plain text search of each feature-computing function's source (a hit here does NOT by itself mean the column is used -- see note below):")
for name, hits in all_clean.items():
    print(f"    {name:35s} {hits if hits else 'CLEAN'}")
print("    create_balance_features' only hit is inside its DOCSTRING (a warning against using newbalanceOrig/newbalanceDest), not in executable code --")
print("    confirmed by reading the function body: it references only df['oldbalanceOrg'] and df['oldbalanceDest'].")
print("    Conclusion: isFraud, isFlaggedFraud, newbalanceOrig, and newbalanceDest are not used as inputs to any feature calculation.")

# 7) No model feature contains raw nameOrig/nameDest
check7 = ("nameOrig" not in model_cols) and ("nameDest" not in model_cols)
print(f"\n[7] Raw nameOrig/nameDest excluded from model feature columns: {check7}")

## 10b. Same-step causality validation (controlled test)

Section 3 established that rows sharing the same `step` have no reliable sub-order, so the design uses STRICTLY EARLIER steps only. This is a claim about behavior, not just intent -- so it needs a direct test, not just a code read.

**Controlled setup:** sender `A` has two transactions at the same `step` (10), then one more transaction at a later step (15). If the same-step exclusion works correctly:
- **Both** step-10 transactions must see **zero** prior history from each other (`prior_sender_transaction_count == 0`, `prior_sender_total_amount == 0`) -- neither one may "see" the other, regardless of row order in the file.
- The step-15 transaction must see **both** step-10 transactions (`prior_sender_transaction_count == 2`, `prior_sender_total_amount == 300`).
- The step-10 transactions' velocity features (`sender_transactions_previous_1_step`, etc.) must also be 0 -- they must not count each other.

The same setup is mirrored for a receiver: account `Z` receives from two different senders (`B`, `C`) at the same step (20), then from a third sender (`D`) at a later step (25).

In [ ]:
toy = pd.DataFrame({
    "step":            [10,    10,    15,    20,   20,   25],
    "type":            ["TRANSFER", "CASH_OUT", "TRANSFER", "PAYMENT", "PAYMENT", "PAYMENT"],
    "amount":          [100.0, 200.0, 300.0, 50.0, 60.0, 70.0],
    "nameOrig":        ["A",   "A",   "A",   "B",  "C",  "D"],
    "oldbalanceOrg":   [1000.0, 900.0, 700.0, 500.0, 500.0, 500.0],
    "newbalanceOrig":  [900.0, 700.0, 400.0, 450.0, 440.0, 430.0],
    "nameDest":        ["X",   "Y",   "X",   "Z",  "Z",  "Z"],
    "oldbalanceDest":  [0.0,   0.0,   100.0, 0.0,  50.0, 110.0],
    "newbalanceDest":  [100.0, 200.0, 400.0, 50.0, 110.0, 180.0],
    "isFraud":         [0, 0, 0, 0, 0, 0],
    "isFlaggedFraud":  [0, 0, 0, 0, 0, 0],
})

toy_feats = build_feature_dataset(toy)
display_cols = [
    "nameOrig", "nameDest", "step", "amount",
    "prior_sender_transaction_count", "prior_sender_total_amount",
    "sender_transactions_previous_1_step", "sender_transactions_previous_6_steps",
    "sender_transactions_previous_24_steps",
    "prior_receiver_transaction_count", "prior_receiver_total_amount",
]
toy_feats[display_cols]

In [ ]:
row_a1, row_a2, row_a3 = toy_feats.loc[0], toy_feats.loc[1], toy_feats.loc[2]
row_b, row_c, row_d = toy_feats.loc[3], toy_feats.loc[4], toy_feats.loc[5]

checks = {
    "Sender A, step10 (txn1): prior_sender_transaction_count == 0": row_a1["prior_sender_transaction_count"] == 0,
    "Sender A, step10 (txn2): prior_sender_transaction_count == 0": row_a2["prior_sender_transaction_count"] == 0,
    "Sender A, step10 (txn1): prior_sender_total_amount == 0": row_a1["prior_sender_total_amount"] == 0,
    "Sender A, step10 (txn2): prior_sender_total_amount == 0": row_a2["prior_sender_total_amount"] == 0,
    "Sender A, step15: prior_sender_transaction_count == 2": row_a3["prior_sender_transaction_count"] == 2,
    "Sender A, step15: prior_sender_total_amount == 300 (100+200)": row_a3["prior_sender_total_amount"] == 300.0,
    "Sender A, step10 (txn1): sender_transactions_previous_1_step == 0 (does not see txn2)": row_a1["sender_transactions_previous_1_step"] == 0,
    "Sender A, step10 (txn2): sender_transactions_previous_1_step == 0 (does not see txn1)": row_a2["sender_transactions_previous_1_step"] == 0,
    "Sender A, step10 (txn1): previous_6/24_steps == 0": row_a1["sender_transactions_previous_6_steps"] == 0 and row_a1["sender_transactions_previous_24_steps"] == 0,
    "Sender A, step10 (txn2): previous_6/24_steps == 0": row_a2["sender_transactions_previous_6_steps"] == 0 and row_a2["sender_transactions_previous_24_steps"] == 0,
    "Receiver Z, step20 (from B): prior_receiver_transaction_count == 0": row_b["prior_receiver_transaction_count"] == 0,
    "Receiver Z, step20 (from C): prior_receiver_transaction_count == 0": row_c["prior_receiver_transaction_count"] == 0,
    "Receiver Z, step20 (from B): prior_receiver_total_amount == 0": row_b["prior_receiver_total_amount"] == 0,
    "Receiver Z, step20 (from C): prior_receiver_total_amount == 0": row_c["prior_receiver_total_amount"] == 0,
    "Receiver Z, step25: prior_receiver_transaction_count == 2": row_d["prior_receiver_transaction_count"] == 2,
    "Receiver Z, step25: prior_receiver_total_amount == 110 (50+60)": row_d["prior_receiver_total_amount"] == 110.0,
}

print("=== SAME-STEP CAUSALITY VALIDATION ===")
all_pass = True
for name, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  [{status}] {name}")
print(f"\nOVERALL: {'PASS' if all_pass else 'FAIL'}")
assert all_pass, "Same-step causality validation FAILED -- see individual checks above."

**Result: PASS on all 16 checks.** Same-step transactions never see each other in either sender-side or receiver-side historical/velocity features -- confirming the `merge_asof(step - offset, direction="backward")` mechanism (section 3) behaves exactly as designed, not just as documented.

## 11. Feature summary

See `src/feature_definitions.py` for the full interview-ready table (name, meaning, source columns, decision-time availability, historical-info usage, leakage risk, dtype) for every one of the below.

In [ ]:
print(f"Total documented features/metadata columns: {len(FEATURE_DEFINITIONS)}")
print(f"Total model-ready feature columns: {len(model_cols)}")
pd.DataFrame([{"name": f.name, "at_decision_time": f.available_at_decision_time, "uses_history": f.uses_historical_info} for f in FEATURE_DEFINITIONS])

## 12. Missing-value checks

In [ ]:
null_counts = feature_df.isnull().sum()
print("Total nulls across the entire feature dataset:", null_counts.sum())
null_counts[null_counts > 0]

## 13. Distribution / sanity checks

In [ ]:
feature_df[model_cols].describe().T

In [ ]:
# Sanity: most senders transact once, so most prior_sender_* features should be 0/no-history for most rows
print("% of rows with prior_sender_has_history == 0 (first-ever transaction for that sender):")
print(f"  {(feature_df['prior_sender_has_history'] == 0).mean() * 100:.2f}%")
print("\n% of rows with prior_receiver_has_history == 0 (first-ever transaction for that receiver):")
print(f"  {(feature_df['prior_receiver_has_history'] == 0).mean() * 100:.2f}%")
print("\n(Consistent with EDA: nameOrig is nearly one-to-one with rows, so most senders have no history; ",
      "nameDest repeats far more, so receiver-history features are populated far more often.)")

## 14. Save the resulting feature dataset

**Size estimate before saving:** 6,362,620 rows x 46 columns, ~2.5 GB in memory. A CSV at this size would land in the 1.5-2 GB range as plain text. We use **Parquet** (columnar, compressed) instead -- requires the `pyarrow` package, installed for this step.

In [ ]:
import os

out_path = "../data/processed/paysim_features.parquet"
feature_df.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")

disk_mb = os.path.getsize(out_path) / 1e6
print(f"Saved: {out_path}")
print(f"Disk size: {disk_mb:,.1f} MB (vs. an estimated ~1.5-2.0 GB if saved as CSV)")

The target `isFraud` is preserved as a column in this saved file (never dropped, never used to build a feature). When loading this file for modeling in later steps, split off `X = df[get_model_feature_columns(df)]` and `y = df["isFraud"]` explicitly, rather than feeding the whole file to a model.